# 학습 데이터셋 EDA — 돌봄부담 진단 모델

`docs/학습데이터셋-컬럼정의.md` 에 기술된 분석을 데이터베이스에서 다시 계산해 시각화한다.
문서의 수치는 **작성 시점의 관측값**이므로, 이 노트북은 그 값을 그대로 옮기지 않고 재산출한 뒤
문서값과 나란히 놓아 **일치 여부까지 확인**한다.

| 항목 | 값 |
|---|---|
| 데이터베이스 | `ABC8pioneer3` (MariaDB) |
| 주 대상 객체 | `cb_dataset_v1` (메타 4 + target 1 + 설명변수 38) |
| 원천 테이블 | `2024_care_burden_std09` (2024 발달장애인 일과 삶 실태조사, 3,000가구) |
| 근거 문서 | `docs/학습데이터셋-컬럼정의.md` · `specs/001-care-burden-map/spec.md` |

## 목차

1. 연결 · 데이터 적재
2. target 분포 — 5구간 × train/test
3. 설명력(MI/H) 38개 순위
4. 결측 구조와 조건부 문항
5. 사례 A — 보조 돌봄제공자 역설 (문서 5.1)
6. 사례 B — 가구주 유형 8범주 (문서 5.2)
7. 배제 6변수 vs 잔존 38변수

### 척도 방향 — 반드시 먼저 읽을 것

`care_burden` 은 **1이 최고부담, 5가 부담 없음**이다. 숫자가 작을수록 부담이 크다.
평균값 비교에서 부호를 반대로 읽기 쉬우므로, 이 노트북의 모든 축에는 부담이 커지는 방향을 화살표로 표시했다.


## 1. 연결 · 데이터 적재

### 1.1 접속 정보

접속 정보는 **프로젝트 루트의 `.env` 파일**에서 읽는다. 이 노트북에는 자격증명을 적지 않는다 —
노트북은 팀원·심사자에게 전달되는 파일이므로 값이 박혀 있으면 전달할 때마다 함께 나간다.

처음 실행한다면 `.env.example` 을 복사해 값을 채운다.

```bash
cp .env.example .env      # 그 뒤 편집기로 열어 값 입력
pip install pandas numpy matplotlib sqlalchemy pymysql
```

이미 셸 환경변수에 같은 이름이 설정돼 있으면 그쪽이 우선한다(`.env` 는 비어 있는 값만 채운다).


In [ ]:
# =====================================================================
# CONFIG — 데이터베이스 접속 정보
#   값은 프로젝트 루트의 .env 에서 읽는다. 이 셀에는 값을 적지 않는다.
#   우선순위 : 셸 환경변수 > .env 파일
# =====================================================================
import os
from pathlib import Path

def load_env(filename='.env'):
    """현재 폴더에서 위로 올라가며 .env 를 찾아 os.environ 을 채운다.
    이미 설정된 환경변수는 덮어쓰지 않는다. 외부 패키지를 쓰지 않는다."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        p = d / filename
        if p.is_file():
            for line in p.read_text(encoding='utf-8').splitlines():
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('\'"'))
            return p
    return None

_env = load_env()
print('.env :', _env if _env else '못 찾음 — 셸 환경변수로 넘기거나 .env.example 을 복사하세요')

DB_HOST     = os.getenv('DB_HOST',     '')
DB_PORT     = int(os.getenv('DB_PORT', '3306'))
DB_USER     = os.getenv('DB_USER',     '')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME     = os.getenv('DB_NAME',     '')
DB_CHARSET  = os.getenv('DB_CHARSET',  'utf8mb4')

# 조회 대상 객체
T_DATASET = 'cb_dataset_v1'                # 정제 데이터셋 (설명변수 38 + target + 메타)
T_META    = 'cb_feature_meta_v1'           # 변수 메타 (있으면 사용, 없으면 건너뜀)
T_RAW     = '2024_care_burden_std09'       # 원천 테이블 (배제 6변수 조회용)

_missing = [k for k, v in (('DB_HOST', DB_HOST), ('DB_USER', DB_USER),
                           ('DB_PASSWORD', DB_PASSWORD), ('DB_NAME', DB_NAME)) if not v]
assert not _missing, f'.env 에 다음 값이 없습니다 : {", ".join(_missing)}'

# 비밀번호는 출력하지 않는다
print(f'대상: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [ ]:
# 공통 임포트 · 한글 폰트 · 팔레트
import warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 140)

# ---- 한글 폰트 : 설치된 것 중 첫 번째를 쓴다 -------------------------
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _cand in ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'Malgun Gothic', 'NanumBarunGothic']:
    if _cand in _installed:
        mpl.rcParams['font.family'] = _cand
        break
else:
    print('한글 폰트를 찾지 못했습니다. 축 라벨이 깨지면 NanumGothic 을 설치하세요.')
mpl.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# ---- 팔레트 : dataviz 검증 스크립트 통과값 ---------------------------
#   부담 5구간  : 단일 색상(red) 순차 램프. 진할수록 부담이 크다.
#                 ordinal 검사 PASS (단조 명도 · 인접 명도차 · 밝은 끝 대비 2.13:1 · 색상폭 5도)
#   2계열 범주  : 범주형 슬롯 1·2. adjacent 검사 PASS (CVD dE 24.7 / 정상시야 dE 33.6)
#   순차 blue   : 결측률 히트맵용. ordinal 검사 PASS
BURDEN_RAMP = {1:'#801e1e', 2:'#a92e2d', 3:'#d04442', 4:'#dd7170', 5:'#e99b9a'}  # 1=최고부담(진함)
BURDEN_NAME = {1:'최고부담군', 2:'고부담군', 3:'중간부담군', 4:'저부담군', 5:'부담 없음'}
CAT1, CAT2 = '#2a78d6', '#eb6834'          # 범주형 슬롯 1(blue) · 2(orange)
SEQ_BLUE   = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281']

SURFACE = '#fcfcfb'; INK1 = '#0b0b0b'; INK2 = '#52514e'; GRID = '#e6e5e1'; AXIS = '#c8c7c2'

mpl.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'figure.dpi': 110, 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlecolor': INK1,
    'axes.labelcolor': INK2, 'text.color': INK1,
    'xtick.color': INK2, 'ytick.color': INK2,
})

def style_axes(ax, grid_axis='y'):
    """격자·축은 뒤로 물리고 데이터가 앞에 오게 한다."""
    ax.set_axisbelow(True)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8, zorder=0)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(AXIS); ax.spines[side].set_linewidth(0.8)
    ax.tick_params(length=0)
    return ax

def wrap(s, n=14):
    return '\n'.join(textwrap.wrap(str(s), n))

print('팔레트·폰트 준비 완료 ·', mpl.rcParams['font.family'])


In [ ]:
# 접속 및 적재
engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset={DB_CHARSET}',
    pool_pre_ping=True,
)

with engine.connect() as con:
    print('서버 버전 :', con.execute(text('SELECT VERSION()')).scalar())

df = pd.read_sql(f'SELECT * FROM `{T_DATASET}`', engine)
print(f'{T_DATASET} : {df.shape[0]:,}행 × {df.shape[1]}열')

train = df[df['split'] == 'train'].copy()
test  = df[df['split'] == 'test'].copy()
print(f'  train {len(train):,}  /  test {len(test):,}')

# 변수 메타는 있으면 쓰고 없으면 건너뛴다 (스키마가 환경마다 다를 수 있음)
try:
    meta_db = pd.read_sql(f'SELECT * FROM `{T_META}`', engine)
    print(f'{T_META} : {meta_db.shape[0]}행 × {meta_db.shape[1]}열 — 컬럼 {list(meta_db.columns)}')
except Exception as e:
    meta_db = None
    print(f'{T_META} 조회 실패(무시하고 진행) : {type(e).__name__}')

df.head(3)


### 1.2 변수 메타 — 문서의 7개 블록

아래 표는 `docs/학습데이터셋-컬럼정의.md` 3장의 **블록 재편**을 그대로 옮긴 것이다.
코드북의 카테고리 분류와 별개로, **돌봄부담이 발생하는 메커니즘**에 따라 묶은 축이므로 DB에는 없다.
`mi_doc` 은 문서에 기록된 MI/H 값이며, 3장에서 재계산값과 대조한다.


In [ ]:
# 문서 3장의 블록 재편 + 문서 기록 MI/H (재계산값과 대조하기 위한 기준선)
META_ROWS = [
    ('help_needed_hours', '도움필요시간', 'G6', '1 돌봄 소요 직접', '범주', 8.23),
    ('daily_routine_satisfaction', '일과만족도', 'G8', '1 돌봄 소요 직접', '범주', 3.51),
    ('understands_work_meaning', '근로의미이해정도', 'F1', '2 당사자 기능수준', '범주', 7.98),
    ('can_work_standard_job', '통상근로가능여부', 'F4', '2 당사자 기능수준', '범주', 6.68),
    ('job_ability_mobility', '직업적능력_이동성', 'F2_2', '2 당사자 기능수준', '범주', 2.47),
    ('job_ability_strength', '직업적능력_체력', 'F2_1', '2 당사자 기능수준', '범주', 2.0),
    ('school_helpfulness', '학교교육도움정도', 'B4', '2 당사자 기능수준', '범주', 2.09),
    ('final_school', '최종학교', 'B1', '2 당사자 기능수준', '범주', 1.44),
    ('has_job_skill_cert', '전문기술자격보유여부', 'F3', '2 당사자 기능수준', '범주', 0.68),
    ('completion_status', '이수상태', 'B2', '2 당사자 기능수준', '범주', 0.27),
    ('overall_health', '전반적건강상태', 'G1', '3 당사자 건강', '범주', 3.35),
    ('has_chronic_disease', '만성질병여부', 'G2', '3 당사자 건강', '범주', 1.3),
    ('health_change_yoy', '작년대비건강변화', 'G1_1', '3 당사자 건강', '범주', 0.89),
    ('early_aging', '조기노화여부', 'G3', '3 당사자 건강', '범주', 0.11),
    ('exercises_regularly', '평소운동여부', 'G4', '3 당사자 건강', '범주', 0.1),
    ('is_employed', '취업여부', 'C1', '4 주간 분리시간', '범주', 2.32),
    ('employment_status', '종사상지위', 'C2', '4 주간 분리시간', '범주', 2.54),
    ('severe_dd_job_willingness', '최중증발달장애인취업의사', 'F4_1', '5 취업 이력·의향', '범주', 3.98),
    ('wants_person_employed', '당사자취업희망여부', 'E6', '5 취업 이력·의향', '범주', 2.93),
    ('past_employment_exp', '과거취업경험', 'E1', '5 취업 이력·의향', '범주', 2.74),
    ('last_job_quit_reason', '마지막일자리퇴사이유', 'E4', '5 취업 이력·의향', '범주', 1.15),
    ('past_job_count', '과거일자리개수', 'E2', '5 취업 이력·의향', '범주', 0.56),
    ('wanted_to_stay_at_last_job', '퇴사당시계속근무희망여부', 'E5', '5 취업 이력·의향', '범주', 0.2),
    ('caregiver_age', '주보호자연령', 'I9_2', '6 돌봄 자원', '연속', 5.11),
    ('family_support_for_employment', '취업에대한가족지지정도', 'F5', '6 돌봄 자원', '범주', 3.94),
    ('household_head_type', '가구주 유형', 'I4', '6 돌봄 자원', '범주', 1.67),
    ('relation_to_person', '당사자와의관계', 'A1', '6 돌봄 자원', '범주', 1.18),
    ('primary_caregiver_type', '주보호자유형', 'I8_1', '6 돌봄 자원', '범주', 0.96),
    ('household_size', '가구원수', 'I1', '6 돌봄 자원', '연속', 0.92),
    ('lives_with_mother', '어머니동거여부', 'I2_1_2', '6 돌봄 자원', '범주', 0.61),
    ('lives_with_father', '아버지동거여부', 'I2_1_1', '6 돌봄 자원', '범주', 0.46),
    ('secondary_caregiver_type', '부보호자유형', 'I8_2', '6 돌봄 자원', '범주', 0.44),
    ('lives_with_person', '당사자동거여부', 'A2', '6 돌봄 자원', '범주', 0.38),
    ('caregiver_gender', '주보호자성별', 'I9_1', '6 돌봄 자원', '범주', 0.21),
    ('caregiver_employment_status', '주보호자취업상태', 'I9_4', '6 돌봄 자원', '범주', 0.2),
    ('age_disability_suspected', '장애의심시작나이', 'A6', '7 생애주기·인구', '연속', 4.39),
    ('person_marital_status', '당사자혼인상태', 'A5', '7 생애주기·인구', '범주', 0.3),
    ('person_gender', '당사자성별', 'A3', '7 생애주기·인구', '범주', 0.03),
]
meta = pd.DataFrame(META_ROWS, columns=['col','label','item','block','kind','mi_doc'])

# FR-004b 로 배제된 6변수 (cb_dataset_v1 에는 없고 원천 테이블에만 있다)
EXCL_ROWS = [
    ('caregiver_life_satisfaction', '보호자삶만족도', 'I14', 19.1, 'target과 동일 구성개념'),
    ('care_difficulty_top1', '돌봄어려움1순위', 'I10', 6.9, '부담을 전제하고 원인을 물음'),
    ('needed_care_service_type', '필요한통합돌봄서비스유형', 'I12_1', 5.0, '부담 판단의 결과'),
    ('work_care_gap_hours', '월평균돌봄공백시간', 'I11_H', 1.1, 'target과 동일 I 블록'),
    ('work_care_gap_exp', '근로중돌봄공백경험', 'I11', 0.6, 'target과 동일 I 블록'),
    ('integrated_care_awareness', '통합돌봄제도인지도', 'I12', 0.4, 'target과 동일 I 블록'),
]
excl = pd.DataFrame(EXCL_ROWS, columns=['col','label','item','mi_doc','reason'])

print(f'설명변수 {len(meta)}개 · 배제 {len(excl)}개')
missing_in_db = [c for c in meta["col"] if c not in df.columns]
if missing_in_db:
    print('경고 — 데이터셋에 없는 컬럼:', missing_in_db)
meta.groupby('block').agg(변수수=('col','size'), 문서MI합=('mi_doc','sum')).round(2)


## 2. target 분포 — 5구간 × train/test

두 가지를 본다.

- **불균형의 크기** — 5단계(부담 없음)가 몇 건인지가 SC-004(macro F1)의 난이도를 좌우한다.
- **층화 분할이 유지됐는지** — train 과 test 의 구간 비율이 어긋나면 최종 평가(SC-004·SC-005)가 왜곡된다.

건수는 규모를, 비율은 층화 성립 여부를 답한다. 두 질문이 다르므로 **축이 다른 두 차트로 나눈다**(한 축에 겹치지 않는다).


In [ ]:
burden_ct = (df.groupby(['care_burden', 'split']).size().unstack('split')
               .reindex(index=[1,2,3,4,5], columns=['train','test']).fillna(0).astype(int))
burden_pct = burden_ct.div(burden_ct.sum(axis=0), axis=1) * 100

tbl = burden_ct.copy()
tbl.insert(0, '구간', [BURDEN_NAME[i] for i in tbl.index])
tbl['train %'] = burden_pct['train'].round(1)
tbl['test %']  = burden_pct['test'].round(1)
tbl['차이 %p']  = (burden_pct['test'] - burden_pct['train']).round(2)
display(tbl)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.4))
names = [BURDEN_NAME[i] for i in [1,2,3,4,5]]
cols  = [BURDEN_RAMP[i] for i in [1,2,3,4,5]]
x = np.arange(5)

# (좌) 건수 — train/test 를 나란히. 규모를 본다.
ax = style_axes(axes[0])
b1 = ax.bar(x - 0.20, burden_ct['train'], 0.38, color=cols, zorder=3)
b2 = ax.bar(x + 0.20, burden_ct['test'],  0.38, color=cols, alpha=0.45, zorder=3,
            edgecolor=SURFACE, linewidth=2)
for rects in (b1, b2):
    for r in rects:
        ax.annotate(f'{int(r.get_height()):,}', (r.get_x() + r.get_width()/2, r.get_height()),
                    ha='center', va='bottom', fontsize=8.5, color=INK2)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
ax.set_title('구간별 건수 — 진한 막대 train, 옅은 막대 test', loc='left')
ax.set_ylabel('가구 수')
ax.legend([b1, b2], ['train (2,398)', 'test (602)'], frameon=False, loc='upper right', fontsize=9)

# (우) 비율 — 층화가 유지됐는지. 두 점을 잇는 선으로 차이를 직접 보여준다.
ax = style_axes(axes[1])
for i, lv in enumerate([1,2,3,4,5]):
    ax.plot([burden_pct.loc[lv,'train'], burden_pct.loc[lv,'test']], [i, i],
            color=GRID, linewidth=3, solid_capstyle='round', zorder=2)
    ax.scatter(burden_pct.loc[lv,'train'], i, s=90, color=BURDEN_RAMP[lv], zorder=4,
               edgecolor=SURFACE, linewidth=2, label='train' if i == 0 else None)
    ax.scatter(burden_pct.loc[lv,'test'], i, s=90, facecolor=SURFACE, zorder=4,
               edgecolor=BURDEN_RAMP[lv], linewidth=2.2, label='test' if i == 0 else None)
    d = burden_pct.loc[lv,'test'] - burden_pct.loc[lv,'train']
    ax.annotate(f'{d:+.2f}%p', (max(burden_pct.loc[lv]) + 1.2, i), va='center', fontsize=8.5, color=INK2)
ax.set_yticks(range(5)); ax.set_yticklabels(names, fontsize=9); ax.invert_yaxis()
ax.set_xlim(0, burden_pct.values.max() + 9)
ax.set_xlabel('구성 비율 (%)')
ax.set_title('층화 분할 점검 — train ● vs test ○', loc='left')
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.grid(axis='y', visible=False); ax.grid(axis='x', color=GRID, linewidth=0.8)

fig.suptitle('돌봄부담 target 분포  ·  1=최고부담 → 5=부담 없음', x=0.005, ha='left',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

print(f"고부담군(1~2단계) 비율 : train {burden_pct.loc[[1,2],'train'].sum():.1f}%  "
      f"test {burden_pct.loc[[1,2],'test'].sum():.1f}%")
print(f"최대 층화 오차 : {(burden_pct['test'] - burden_pct['train']).abs().max():.2f}%p")


## 3. 설명력(MI/H) 38개 순위

`MI/H` 는 상호정보량을 target 엔트로피로 나눈 값이다. **target 불확실성의 몇 %를 그 변수 하나가 설명하는가**를 뜻한다.

두 가지 계산 규칙을 문서와 맞춘다.

- **train 2,398건만 사용한다.** test 를 섞으면 SC-004·SC-005 가 낙관적으로 나온다.
- **결측을 별도 범주로 둔다.** 상위 변수들의 결측은 누락이 아니라 조건부 문항의 "해당 없음"이며, 그 자체가 정보다(FR-004d).

연속형 3개(`caregiver_age` 81레벨, `age_disability_suspected` 60레벨, `household_size` 9레벨)는
레벨 수가 많아 플러그인 추정량이 부풀려진다. **10분위로 구간화**한 뒤 계산하고, 문서값과의 차이를 함께 표시한다.


In [ ]:
def mi_over_h(x: pd.Series, y: pd.Series, n_bins=None) -> float:
    """MI(X;Y) / H(Y). 결측은 버리지 않고 '__NA__' 라는 하나의 범주로 취급한다."""
    if n_bins is not None and x.notna().sum() > 0:
        # 연속형은 분위 구간화. 중복 경계는 합쳐서 레벨 수 폭증을 막는다.
        x = pd.qcut(x.astype(float), q=n_bins, duplicates='drop').astype(object)
    xs = x.where(x.notna(), '__NA__').astype(str)
    ys = y.astype(str)
    n  = len(ys)
    pxy = pd.crosstab(xs, ys) / n
    px  = pxy.sum(axis=1)
    py  = pxy.sum(axis=0)
    nz  = pxy.values > 0
    outer = np.outer(px.values, py.values)
    mi  = float((pxy.values[nz] * np.log(pxy.values[nz] / outer[nz])).sum())
    hy  = float(-(py * np.log(py)).sum())
    return mi / hy * 100 if hy > 0 else np.nan

y_tr = train['care_burden']
rows = []
for _, r in meta.iterrows():
    if r['col'] not in train.columns:
        continue
    bins = 10 if r['kind'] == '연속' else None
    rows.append({**r.to_dict(), 'mi_calc': mi_over_h(train[r['col']], y_tr, n_bins=bins)})
mi_df = pd.DataFrame(rows).sort_values('mi_calc', ascending=False).reset_index(drop=True)
mi_df['순위'] = mi_df.index + 1
mi_df['문서대비'] = (mi_df['mi_calc'] - mi_df['mi_doc']).round(2)

display(mi_df[['순위','col','label','block','kind','mi_calc','mi_doc','문서대비']]
        .round({'mi_calc': 2}).head(15))


In [ ]:
# (1) 38개 순위 — 단일 색상. 값 자체가 크기이므로 색으로 순위를 다시 칠하지 않는다.
d = mi_df.sort_values('mi_calc')
fig, ax = plt.subplots(figsize=(9.6, 10.4))
style_axes(ax, grid_axis='x')
ypos = np.arange(len(d))
ax.barh(ypos, d['mi_calc'], height=0.68, color=CAT1, zorder=3)
ax.scatter(d['mi_doc'], ypos, s=26, facecolor=SURFACE, edgecolor=INK2, linewidth=1.3, zorder=5,
           label='문서 기록값')
for i, (v, lab) in enumerate(zip(d['mi_calc'], d['label'])):
    ax.annotate(f'{v:.2f}', (v + 0.12, i), va='center', fontsize=8, color=INK2)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{l}  · {b.split()[0]}" for l, b in zip(d['label'], d['block'])], fontsize=8.6)
ax.set_xlabel('MI / H  (target 불확실성의 설명 비율, %)')
ax.set_xlim(0, max(d['mi_calc'].max(), d['mi_doc'].max()) * 1.16)
ax.set_title('설명변수 38개 단독 설명력 — 막대 재계산값 · ○ 문서 기록값', loc='left')
ax.legend(frameon=False, loc='lower right', fontsize=9)
# 1% 기준선 : 문서 5장의 하위 17개 구분선
ax.axvline(1.0, color=CAT2, linewidth=1.4, linestyle=(0, (4, 3)), zorder=2)
ax.annotate('1%', (1.0, len(d) - 0.4), color=CAT2, fontsize=9, ha='center', va='bottom')
fig.tight_layout(); plt.show()

# (2) 블록별 합계 — 어느 축이 설명력을 많이 지고 있는가
blk = (mi_df.groupby('block').agg(합계=('mi_calc','sum'), 변수수=('col','size'))
            .sort_values('합계', ascending=True))
fig, ax = plt.subplots(figsize=(9.6, 3.8))
style_axes(ax, grid_axis='x')
ax.barh(blk.index, blk['합계'], height=0.62, color=CAT1, zorder=3)
for i, (v, n) in enumerate(zip(blk['합계'], blk['변수수'])):
    ax.annotate(f'{v:.1f}%  ({n}개)', (v + 0.25, i), va='center', fontsize=9, color=INK2)
ax.set_xlim(0, blk['합계'].max() * 1.28)
ax.set_xlabel('블록 내 MI/H 합계 (%)')
ax.set_title('7개 블록별 설명력 합계 — 변수 수가 많다고 설명력이 큰 것은 아니다', loc='left')
fig.tight_layout(); plt.show()

print('상호정보량은 변수 하나씩만 보는 지표다. 조합 효과와 중복을 반영하지 못하므로')
print('FR-004c 의 최종 문항 집합은 이 순위가 아니라 38개 전체 모델 대비 성능 손실 기준으로 정한다.')


## 4. 결측 구조와 조건부 문항

이 데이터셋의 결측은 대부분 **데이터 누락이 아니라 분기 문항의 "해당 없음"**이다.
예를 들어 `past_job_count`(과거 일자리 개수)의 결측 88%는 과거 취업 경험이 없어 문항 자체가 제시되지 않은 응답이다.

결측률이 **비슷한 값으로 뭉치면 같은 분기 조건을 공유한다**는 신호다. 이 군집이 FR-004d 의
"해당사항 없음" 선택지 설계와 그대로 대응하므로, 값이 아니라 **뭉침**을 보는 것이 이 차트의 목적이다.


In [ ]:
miss = (train[[c for c in meta['col'] if c in train.columns]].isna().mean() * 100)
miss = (meta.set_index('col').join(miss.rename('missing_pct'))
            .sort_values('missing_pct', ascending=True).reset_index())

# 25% 를 넘으면 조건부 문항으로 본다 (문서: 상위 8개 변수의 결측은 분기 미해당)
is_cond = miss['missing_pct'] > 25
colors  = np.where(is_cond, CAT2, CAT1)

fig, ax = plt.subplots(figsize=(9.6, 10.4))
style_axes(ax, grid_axis='x')
ypos = np.arange(len(miss))
ax.barh(ypos, miss['missing_pct'], height=0.68, color=colors, zorder=3)
for i, v in enumerate(miss['missing_pct']):
    if v > 0.005:
        ax.annotate(f'{v:.1f}', (v + 0.9, i), va='center', fontsize=8, color=INK2)
ax.set_yticks(ypos); ax.set_yticklabels(miss['label'], fontsize=8.6)
ax.set_xlabel('결측률 (%, train 2,398건)')
ax.set_xlim(0, 100)
ax.axvline(25, color=AXIS, linewidth=1.2, linestyle=(0, (4, 3)), zorder=2)
handles = [plt.Rectangle((0,0),1,1, color=CAT2), plt.Rectangle((0,0),1,1, color=CAT1)]
ax.legend(handles, ['조건부 문항 (>25%) — "해당 없음"', '전원 응답 문항'],
          frameon=False, loc='lower right', fontsize=9)
ax.set_title('결측률 — 값이 아니라 뭉치는 지점을 본다', loc='left')
fig.tight_layout(); plt.show()

# 결측률이 같은 값으로 뭉치는 군집 = 같은 분기 조건
grp = (miss[is_cond].assign(rounded=lambda t: t['missing_pct'].round(1))
                    .groupby('rounded')['label'].apply(list))
print('같은 결측률로 묶이는 군집 (같은 분기 조건을 공유할 가능성):')
for k, v in grp.items():
    if len(v) > 1:
        print(f'  {k:5.1f}% — {", ".join(v)}')


## 5. 사례 A — 보조 돌봄제공자 역설 (문서 5.1)

`secondary_caregiver_type = 9` 는 결측이 아니라 **"보조 돌봄제공자 없음"이라는 실제 범주**다.
이 값을 기준으로 나누면 통념과 어긋나는 관측이 나온다 — **돌봄 자원이 없는 쪽의 부담이 오히려 낮다.**

`care_burden` 은 작을수록 부담이 크므로, "없음" 집단의 평균이 **더 크면 부담이 더 적다**는 뜻이다.
부호를 반대로 읽기 쉬운 지점이라 아래 차트는 평균 대신 **고부담군(1~2단계) 비율**을 함께 놓는다.


In [ ]:
sec = train[train['secondary_caregiver_type'].notna()].copy()
sec['그룹'] = np.where(sec['secondary_caregiver_type'].astype(float) == 9,
                      '없음 (값 9)', '있음 (값 1~8)')

summ = (sec.groupby('그룹')
           .agg(n=('care_burden','size'),
                평균부담=('care_burden','mean'),
                고부담군비율=('care_burden', lambda s: (s <= 2).mean()*100))
           .reindex(['없음 (값 9)', '있음 (값 1~8)']))
display(summ.round({'평균부담': 3, '고부담군비율': 1}))

dist = (sec.groupby(['그룹','care_burden']).size().unstack('care_burden')
           .reindex(index=['없음 (값 9)','있음 (값 1~8)'], columns=[1,2,3,4,5]).fillna(0))
distp = dist.div(dist.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(12.6, 3.9), gridspec_kw={'width_ratios': [2.1, 1]})

# (좌) 구간 구성 100% 누적 — 분포 전체를 본다. 조각 사이 2px 여백으로 경계를 세운다.
ax = style_axes(axes[0], grid_axis='x')
left = np.zeros(len(distp))
for lv in [1,2,3,4,5]:
    ax.barh(distp.index, distp[lv], left=left, height=0.55, color=BURDEN_RAMP[lv],
            edgecolor=SURFACE, linewidth=2, zorder=3, label=BURDEN_NAME[lv])
    for i, v in enumerate(distp[lv]):
        if v >= 6:
            ax.annotate(f'{v:.0f}', (left[i] + v/2, i), ha='center', va='center',
                        fontsize=8.5, color='white' if lv <= 2 else INK1)
    left += distp[lv].values
ax.set_xlim(0, 100); ax.set_xlabel('구성 비율 (%)')
ax.invert_yaxis()
ax.set_title('부담 구간 구성 — 왼쪽이 고부담', loc='left')
ax.legend(frameon=False, ncol=5, fontsize=8.6, loc='upper center', bbox_to_anchor=(0.5, -0.22))

# (우) 고부담군 비율 — 결론에 해당하는 한 숫자
ax = style_axes(axes[1])
bars = ax.bar(summ.index, summ['고부담군비율'], width=0.5,
              color=[BURDEN_RAMP[3], BURDEN_RAMP[2]], zorder=3)
for r, (n, v) in zip(bars, zip(summ['n'], summ['고부담군비율'])):
    ax.annotate(f'{v:.1f}%\n(n={n:,})', (r.get_x()+r.get_width()/2, v), ha='center',
                va='bottom', fontsize=9, color=INK2)
ax.set_ylim(0, max(summ['고부담군비율'])*1.32)
ax.set_ylabel('고부담군(1~2) 비율 (%)')
ax.set_xticklabels([wrap(s, 10) for s in summ.index], fontsize=9)
ax.set_title('통념과 반대 방향', loc='left')
fig.tight_layout(); plt.show()

gap = summ.loc['있음 (값 1~8)','고부담군비율'] - summ.loc['없음 (값 9)','고부담군비율']
print(f'고부담군 비율 차이 : 있음이 {gap:+.1f}%p 높다')
print()
print('해석 — 역인과로 보는 것이 자연스럽다. 부담이 큰 가구일수록 보조 돌봄제공자가 투입되는 것이지,')
print('보조 제공자가 없어서 부담이 커지는 것이 아니다. 이 변수는 부담의 원인이 아니라 결과에 가깝다.')
print('FR-011 의 자연어 변환에서 "보조 돌봄제공자가 있어서 부담이 크다"로 읽히지 않도록 문구를 검토해야 한다.')


## 6. 사례 B — 가구주 유형 8범주 (문서 5.2)

원천 컬럼명은 `is_household_head` 였으나 이진 플래그가 아니라 **가구주가 누구인지를 나타내는 8개 범주**다.
`cb_dataset_v1` 에서 `household_head_type` 으로 개명했다.

단독 설명력은 1.67%로 중위권이지만 **범주마다 해석이 명확해 SHAP 설명 문장으로 옮기기 좋은 변수**다(FR-011).

부담과 취업률은 **단위가 다른 두 측정값**이므로 한 축에 겹치지 않고 나란한 세 패널로 나눈다.


In [ ]:
HEAD = {1:'아버지', 2:'어머니', 3:'장애인 당사자', 4:'형제자매',
        5:'조부모', 6:'배우자', 7:'자녀', 8:'기타'}

hh = train[train['household_head_type'].notna()].copy()
hh['가구주'] = hh['household_head_type'].astype(float).astype(int).map(HEAD)

agg = (hh.groupby('가구주')
         .agg(n=('care_burden','size'),
              평균부담=('care_burden','mean'),
              고부담군비율=('care_burden', lambda s: (s <= 2).mean()*100),
              취업률=('is_employed', lambda s: (s.astype(float) == 1).mean()*100))
         .sort_values('고부담군비율'))
display(agg.round({'평균부담': 3, '고부담군비율': 1, '취업률': 1}))

fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6), sharey=True)
ypos = np.arange(len(agg))
# 부담 크기에 따라 램프를 매기되, 색은 순위가 아니라 '평균 부담 구간'을 따른다
cols = [BURDEN_RAMP[int(np.clip(round(v), 1, 5))] for v in agg['평균부담']]

panels = [
    ('고부담군비율', '고부담군(1~2) 비율 (%)', '고부담군 비율 — 높을수록 부담 큼', '{:.1f}%'),
    ('평균부담',     '평균 care_burden',        '평균 부담 — ← 왼쪽일수록 부담 큼', '{:.2f}'),
    ('취업률',       '당사자 취업률 (%)',        '당사자 취업률 — 분리 시간의 대리 지표', '{:.1f}%'),
]
for ax, (col, xlab, title, fmt) in zip(axes, panels):
    style_axes(ax, grid_axis='x')
    ax.barh(ypos, agg[col], height=0.66, color=cols, zorder=3)
    span = agg[col].max() - min(agg[col].min(), 0)
    for i, v in enumerate(agg[col]):
        ax.annotate(fmt.format(v), (v + span*0.03, i), va='center', fontsize=8.6, color=INK2)
    ax.set_xlabel(xlab); ax.set_title(title, loc='left', fontsize=10.5)
    ax.set_xlim(0, agg[col].max() * 1.20)

# 평균 부담만 축을 뒤집어 '오른쪽 = 부담 큼' 으로 방향을 통일한다
axes[1].invert_xaxis()
axes[1].set_xlim(agg['평균부담'].max() * 1.06, agg['평균부담'].min() * 0.90)

axes[0].set_yticks(ypos)
axes[0].set_yticklabels([f'{k}  (n={n:,})' for k, n in zip(agg.index, agg['n'])], fontsize=9)
fig.suptitle('가구주 유형별 돌봄부담과 당사자 취업률  ·  train 2,398건',
             x=0.005, ha='left', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()

lo, hi = agg.index[0], agg.index[-1]
print(f'가장 부담이 낮은 유형 : {lo} — 고부담군 {agg.loc[lo,"고부담군비율"]:.1f}%, 취업률 {agg.loc[lo,"취업률"]:.1f}%')
print(f'가장 부담이 높은 유형 : {hi} — 고부담군 {agg.loc[hi,"고부담군비율"]:.1f}%, 취업률 {agg.loc[hi,"취업률"]:.1f}%')


## 7. 배제 6변수 vs 잔존 38변수

target `care_burden` 은 원 문항 **I13** 이다. 아래 6개는 같은 I 블록에 속하거나 동일 구성개념을 측정하므로
FR-004b 에 따라 문항 모집단과 모델 입력 양쪽에서 전면 배제했다.

배제 비용은 작지 않다. **설명력 1위 변수가 여기 포함되어 있어** 잔존 38개의 최상위 설명력이 크게 떨어진다.
그럼에도 배제한 것은, 이 변수들을 쓰면 모델 성능은 오르지만 **진단 도구로서는 무의미해지기** 때문이다.

배제 변수는 `cb_dataset_v1` 에 없으므로 원천 테이블에서 직접 조회한다. 실패하면 문서 기록값으로 대체한다.


In [ ]:
# 배제 변수는 원천 테이블에만 있다. train 행만 골라 같은 기준으로 계산한다.
try:
    cols_sql = ', '.join(f'`{c}`' for c in excl['col'])
    raw = pd.read_sql(f'SELECT `care_burden`, {cols_sql} FROM `{T_RAW}`', engine)
    # 원천에는 split 이 없으므로 전체 3,000건 기준으로 계산한다 (문서의 '전체 MI/H' 와 동일 기준)
    raw = raw.replace({' ': np.nan, '': np.nan})
    excl['mi_calc'] = [mi_over_h(raw[c], raw['care_burden']) for c in excl['col']]
    basis = '재계산 (원천 3,000건)'
except Exception as e:
    excl['mi_calc'] = excl['mi_doc']
    basis = f'문서 기록값 대체 ({type(e).__name__})'
    print(f'원천 테이블 조회 실패 — {basis}')

comp = pd.concat([
    excl[['col','label','mi_calc']].assign(구분='배제 (FR-004b)'),
    mi_df.nlargest(8, 'mi_calc')[['col','label','mi_calc']].assign(구분='잔존 상위 8'),
]).sort_values('mi_calc')

fig, ax = plt.subplots(figsize=(9.6, 5.4))
style_axes(ax, grid_axis='x')
ypos = np.arange(len(comp))
cols_ = np.where(comp['구분'] == '배제 (FR-004b)', CAT2, CAT1)
ax.barh(ypos, comp['mi_calc'], height=0.66, color=cols_, zorder=3)
for i, v in enumerate(comp['mi_calc']):
    ax.annotate(f'{v:.2f}', (v + 0.25, i), va='center', fontsize=8.8, color=INK2)
ax.set_yticks(ypos); ax.set_yticklabels(comp['label'], fontsize=9)
ax.set_xlabel(f'MI / H (%)  ·  {basis}')
ax.set_xlim(0, comp['mi_calc'].max() * 1.16)
handles = [plt.Rectangle((0,0),1,1, color=CAT2), plt.Rectangle((0,0),1,1, color=CAT1)]
ax.legend(handles, ['배제 — Data Leakage', '잔존 상위 8'], frameon=False, loc='lower right', fontsize=9)
ax.set_title('배제한 설명력과 남긴 설명력', loc='left')
fig.tight_layout(); plt.show()

top_excl = excl.nlargest(1, 'mi_calc').iloc[0]
top_keep = mi_df.iloc[0]
print(f"배제 최상위 : {top_excl['label']} ({top_excl['item']}) {top_excl['mi_calc']:.2f}%")
print(f"잔존 최상위 : {top_keep['label']} ({top_keep['item']}) {top_keep['mi_calc']:.2f}%")
print(f"배제로 잃은 최상위 설명력 : {top_excl['mi_calc'] - top_keep['mi_calc']:.2f}%p")
display(excl[['item','label','mi_calc','mi_doc','reason']].round({'mi_calc': 2}))


---

## 정리

| 섹션 | 확인한 것 | 후속 작업 |
|---|---|---|
| 2 | 층화 분할이 유지됐는지, 5단계 표본이 얼마나 적은지 | SC-004 macro F1 의 난이도 산정 |
| 3 | 단독 설명력 순위와 블록별 분포 | FR-004c 최소 문항 집합 도출의 **참고 자료** (선별 기준 아님) |
| 4 | 결측이 분기 구조에서 나온다는 것 | FR-004d "해당사항 없음" 선택지 설계 |
| 5 | 보조 돌봄제공자가 부담의 **결과**에 가깝다는 것 | FR-011 자연어 변환에서 인과 표현 회피 |
| 6 | 가구주 유형이 해석 가능한 범주 구조를 갖는다는 것 | FR-011 설명 문장의 우선 활용 후보 |
| 7 | 배제 비용의 크기 | SC-004·SC-005 미달 시 배제 규칙을 되돌리지 않는다는 결정의 근거 |

### 이 노트북이 하지 않는 것

- **test 를 쓰지 않는다.** `split='test'` 602건은 문항 선별·판정 불가 기준 보정·튜닝·모델 선택이
  모두 끝난 뒤 단 한 번만 사용한다. 2장의 층화 점검에서만 비율을 확인했고 값을 의사결정에 쓰지 않았다.
- **인과를 주장하지 않는다.** 5장의 역설이 보여주듯 이 데이터의 상관은 역인과를 포함한다.
